# 01 — Interactive Inference (single cell)

Loads Qwen3-VL and runs the three input modes (`text`, `image`, `image+text`) on one dataset, caching per-sample JSON records with `pred_labels`, `referred_words`, `referred_image_concepts`, and the full `input_ids` / `output_ids`.

For grid runs across models and datasets, use `run_inference.py` and `run_all.sh` instead.

Diagnostics (parse rate, hF1 distribution, low-hF1 inspection) live in `07_analysis.ipynb`. Aggregate hF1 across all caches is computed by the top-level `compute_metrics.py`.


In [ ]:
CONFIG = {
    "dataset_path": "../../datasets/translated/",
    "language": "uk",          # ISO 639-1: affects prompting
    "split": "test",
    "label_column": "labels",
    "text_column": "text",
    "image_column": "image",   # relative filename inside dataset_path/images/
    "model_id": "Qwen/Qwen3-VL-4B-Instruct",
    "max_new_tokens": 300,
    "image_size": (256, 256),
    "output_dir": "./data_translated_no_propaganda_qwen4b/inference_cache",
    "modes": ["text", "image", "image+text"],
    "seed": 42,
}

In [ ]:
import json
import os
import random
import traceback
from pathlib import Path

import numpy as np
import torch
from json_repair import repair_json
from PIL import Image
from tqdm import tqdm
from transformers import AutoProcessor
from transformers.models.qwen3_vl import modeling_qwen3_vl
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent.resolve()))
from src.hierarchical_f1 import hierarchical_f1

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

OUT_DIR = Path(CONFIG["output_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUT_DIR.resolve()}")

In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

ann_path = Path(CONFIG["dataset_path"]) / "annotations" / f"{CONFIG['split']}.jsonl"
dataset = load_jsonl(ann_path)
img_dir = Path(CONFIG["dataset_path"]) / "images"

print(f"Loaded {len(dataset)} samples from {ann_path}")
print(f"Sample keys: {list(dataset[0].keys())}")
print(f"Example: {dataset[0]}")

In [ ]:
UNIQUE_LABELS = [
    "Appeal to (Strong) Emotions",
    "Appeal to authority",
    "Appeal to fear/prejudice",
    "Bandwagon",
    "Black-and-white Fallacy/Dictatorship",
    "Causal Oversimplification",
    "Doubt",
    "Exaggeration/Minimisation",
    "Flag-waving",
    "Glittering generalities (Virtue)",
    "Loaded Language",
    "Misrepresentation of Someone's Position (Straw Man)",
    "Name calling/Labeling",
    "Obfuscation, Intentional vagueness, Confusion",
    "Presenting Irrelevant Data (Red Herring)",
    "Reductio ad hitlerum",
    "Repetition",
    "Slogans",
    "Smears",
    "Thought-terminating cliché",
    "Transfer",
    "Whataboutism",
    # "NO_PROPAGANDA",
]

In [ ]:

LANG_INSTRUCTION = {
    "uk": "All values in referred_words and referred_image_concepts must be in Ukrainian.",
    "en": "All values in referred_words and referred_image_concepts must be in English.",
}

LANG_INSTRUCTION_IMAGE_ONLY = {
    "uk": "All values in referred_image_concepts must be in Ukrainian.",
    "en": "All values in referred_image_concepts must be in English.",
}

BASE_PROMPT_ALTERNATIVE = '''You are an expert in analyzing communication strategies in visual media.
Classify the provided content into one or multiple of the following persuasion techniques :
{unique_labels}
As output, provide JSON with three fields: "labels", "referred_words", and "referred_image_concepts".
- "labels": list of persuasion techniques used, or "NO_PROPAGANDA" if the content doesn't contain persuasion techniques.
- "referred_words": list of the most important INDIVIDUAL words from the content text supporting your classification. Only single words, not phrases. Only key words most relevant to the classification.
- "referred_image_concepts": up to 5 visual elements most strongly supporting your classification, ranked by importance. No duplicates. Empty list if no image. The list could be empty if the image doesn't provide any clues.
Visual elements rules:
- Be concrete: name actual people, objects, symbols. Do NOT write abstract scene descriptions like "people in a stressful situation".
- For visual elements: use the shortest identifying phrase — "Obama"/"Обама", "flag"/"прапор". Do not use words on the image themselves.
Language note: {lang_note}
JSON format output example:
{{
    "labels": ["label1", "label2"],
    "referred_words": ["word1", "word2"],
    "referred_image_concepts": ["concept1", "concept2"]
}}
Text from the content: {text}
Output:
'''
#, or "NO_PROPAGANDA" if the content doesn't contain persuasion techniques.
IMAGE_ONLY_PROMPT_ALTERNATIVE = '''You are an expert in analyzing communication strategies in visual media.
Classify the provided content IMAGE into one or multiple of the following persuasion techniques :
{unique_labels}
As output, provide JSON with two fields: "labels" and "referred_image_concepts".
- "labels": list of persuasion techniques used, or "NO_PROPAGANDA" if the content doesn't contain persuasion techniques.
- "referred_image_concepts": up to 5 visual elements most strongly supporting your classification, ranked by importance. No duplicates. The list could be empty if the image doesn't provide any clues.
Visual elements rules:
- Be concrete: name actual people, objects, symbols. Do NOT write abstract scene descriptions like "people in a stressful situation".
- For words/text visible in the image: do not include ANY of them to referred_image_concepts, as they belong to the text modality. Only visual elements should be included.
- For visual elements: use the shortest identifying phrase — "Obama"/"Обама", "flag"/"прапор". Do not use words on the image themselves.
Language note: {lang_note}
JSON format output example:
{{
    "labels": ["label1", "label2"],
    "referred_image_concepts": ["concept1", "concept2"]
}}
Output:
'''

def build_messages(mode, ocr_text=""):
    lang_note = LANG_INSTRUCTION.get(CONFIG["language"], "")
    img_lang_note = LANG_INSTRUCTION_IMAGE_ONLY.get(CONFIG["language"], "")
    if mode == "text":
        prompt = BASE_PROMPT_ALTERNATIVE.format(
            lang_note=lang_note,
            unique_labels=UNIQUE_LABELS,
            text=ocr_text,
        )
        return [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    elif mode == "image":
        prompt = IMAGE_ONLY_PROMPT_ALTERNATIVE.format(unique_labels=UNIQUE_LABELS, lang_note=img_lang_note,)
        return [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    else:  # image+text
        prompt = BASE_PROMPT_ALTERNATIVE.format(
            lang_note=lang_note,
            unique_labels=UNIQUE_LABELS,
            text=ocr_text,
        )
        return [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]


In [ ]:
model = modeling_qwen3_vl.Qwen3VLForConditionalGeneration.from_pretrained(
    CONFIG["model_id"],
    device_map="cuda",
    dtype=torch.bfloat16,
)
model.eval()
processor = AutoProcessor.from_pretrained(CONFIG["model_id"])
print(f"Model loaded: {CONFIG['model_id']}")

In [ ]:
def run_inference(mode):
    mode_dir = OUT_DIR / mode.replace("+", "_")
    mode_dir.mkdir(parents=True, exist_ok=True)
    manifest = []
    failures = []

    for item in tqdm(dataset, desc=f"Mode={mode}"):
        sample_id = item["id"]
        cache_file = mode_dir / f"{sample_id}.json"

        if cache_file.exists():  # resume support
            manifest.append({"id": sample_id, "path": str(cache_file)})
            continue

        try:
            ocr_text = item.get(CONFIG["text_column"], "")
            image_path = img_dir / item.get(CONFIG["image_column"], "")

            messages = build_messages(mode, ocr_text)
            chat_text = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )

            if mode in ("image", "image+text"):
                image = Image.open(image_path).convert("RGB").resize(CONFIG["image_size"])
                inputs = processor(
                    text=[chat_text], images=[image], padding=True, return_tensors="pt"
                ).to("cuda")
            else:
                inputs = processor(text=[chat_text], padding=True, return_tensors="pt").to("cuda")

            with torch.no_grad():
                generate_kwargs = dict(
                    input_ids=inputs.input_ids,
                    attention_mask=inputs.attention_mask,
                    max_new_tokens=CONFIG["max_new_tokens"],
                    do_sample=False,
                )
                if mode in ("image", "image+text"):
                    generate_kwargs["pixel_values"] = inputs.pixel_values
                    generate_kwargs["image_grid_thw"] = inputs.image_grid_thw

                output_ids = model.generate(**generate_kwargs)

            input_len = inputs.input_ids.shape[1]
            gen_text = processor.decode(output_ids[0, input_len:], skip_special_tokens=True)
            try:
                parsed = json.loads(repair_json(gen_text))
            except Exception:
                parsed = {}

            record = {
                "id": sample_id,
                "mode": mode,
                "gold_labels": item.get(CONFIG["label_column"], []),
                "pred_labels": parsed.get("labels", []),
                "referred_words": parsed.get("referred_words", []),
                "referred_image_concepts": parsed.get("referred_image_concepts", []),
                "generated_text": gen_text,
                "input_ids": inputs.input_ids[0].tolist(),
                "output_ids": output_ids[0].tolist(),
                "input_len": input_len,
                "image_grid_thw": inputs.image_grid_thw[0].tolist() if mode in ("image", "image+text") else None,
                "parse_ok": bool(parsed.get("labels")),
            }

            with open(cache_file, "w", encoding="utf-8") as f:
                json.dump(record, f, ensure_ascii=False, indent=2)

            manifest.append({"id": sample_id, "path": str(cache_file)})

        except Exception as e:
            print(f"  FAIL {sample_id}: {e}")
            traceback.print_exc()
            failures.append({"id": sample_id, "error": str(e)})

        finally:
            torch.cuda.empty_cache()

    with open(mode_dir / "manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)
    with open(mode_dir / "failures.json", "w") as f:
        json.dump(failures, f, indent=2)

    print(f"Mode '{mode}': {len(manifest)} cached, {len(failures)} failures")
    return manifest


In [ ]:
for mode in CONFIG["modes"]:
    run_inference(mode)
print("\nAll modes done. Inference cache ready.")